In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# Source resolution + upstream artifact verification
# ---------------------------------------------------------------
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

required = [
    "results/ml_ad_scores.csv",
    "results/similarity_scores.csv",
    "results/developability_scores.csv",
    "results/ad_results.csv",
]
for rel in required:
    if not (ROOT / rel).exists():
        raise RuntimeError(f"Missing upstream artifact: {rel} — rerun its producer first")

ml_ad = pd.read_csv(ROOT / "results" / "ml_ad_scores.csv")
sim   = pd.read_csv(ROOT / "results" / "similarity_scores.csv")
dev   = pd.read_csv(ROOT / "results" / "developability_scores.csv")
cat   = pd.read_csv(ROOT / "data" / "cultivar_compound_catalogue.csv")

# AD consistency: ml_ad_scores (03) must match ad_results (02)
ad_02 = pd.read_csv(ROOT / "results" / "ad_results.csv")
_chk = ml_ad.merge(ad_02[["compound_id", "ad_distance"]], on="compound_id", suffixes=("_03", "_02"))
_max_diff = (_chk["ad_distance_03"] - _chk["ad_distance_02"]).abs().max()
if _max_diff > 1e-4:
    raise RuntimeError(
        f"AD inconsistency between 03 and 02 (max diff {_max_diff:.6f}). Rerun 03 first."
    )
print(f"Upstream verified: {len(ml_ad)} rosemary compounds, AD max diff {_max_diff:.2e}")

# ---------------------------------------------------------------
# known_active from ChEMBL actives (replaces hard-coded False)
# ---------------------------------------------------------------
splits = json.loads((ROOT / "splits" / "split_indices.json").read_text())
curated = pd.read_csv(ROOT / splits["curated_csv"]).reset_index(drop=True)

train_actives = curated.iloc[splits["indices"]["train"]]
train_actives = train_actives[train_actives["activity_class"] == "active"]
test_actives = curated.iloc[splits["indices"]["test"]]
test_actives = test_actives[test_actives["activity_class"] == "active"]

train_keys = set(train_actives["standard_inchikey"])
test_keys  = set(test_actives["standard_inchikey"])

ik_map = cat.drop_duplicates("compound_id")[["compound_id", "inchikey"]]
df = ml_ad.merge(sim, on="compound_id").merge(dev, on="compound_id").merge(ik_map, on="compound_id", how="left")

df["exact_training_match"] = df["inchikey"].isin(train_keys)
df["exact_test_match"]     = df["inchikey"].isin(test_keys)
df["known_active_flag"]    = df["inchikey"].isin(train_keys | test_keys)
df["known_active"]         = (
    df["exact_training_match"] | df["exact_test_match"] | df["known_active_flag"]
)
df = df.drop(columns=["inchikey"])

# ---------------------------------------------------------------
# Scores
# ---------------------------------------------------------------
df["novelty_bonus"] = np.where(
    df["known_active"], 0.0,
    np.where(df["max_similarity"] < 0.20, 1.0, 0.0),
)

df["final_score"] = (
    0.40 * df["ML_rank_score"]
    + 0.25 * df["similarity_support"]
    + 0.25 * df["developability_score"]
    + 0.10 * df["novelty_bonus"]
)

df["ligand_only_score"] = (
    0.60 * df["similarity_support"]
    + 0.30 * df["developability_score"]
    + 0.10 * df["novelty_bonus"]
)

df["high_ml_ad_extrapolation"] = (df["ad_flag"] == "extrapolation") & (df["ML_rank_score"] >= 0.80)

df = df.sort_values("final_score", ascending=False).reset_index(drop=True)
df["final_rank"] = df.index + 1

df_lig = df.sort_values("ligand_only_score", ascending=False).reset_index(drop=True)
df_lig["ligand_only_rank"] = df_lig.index + 1

df = df.drop(columns=["ligand_only_rank"], errors="ignore").merge(
    df_lig[["compound_id", "ligand_only_rank"]], on="compound_id"
)

final_table = df.merge(cat, on="compound_id")
final_table.to_csv(ROOT / "results" / "screened_compounds_annotated.csv", index=False)

print(f"Ranking locked: {final_table['compound_id'].nunique()} unique compounds "
      f"({len(final_table)} pair rows)")
print(f"  known_active compounds: {int(final_table.drop_duplicates('compound_id')['known_active'].sum())}")

Upstream verified: 28 rosemary compounds, AD max diff 0.00e+00
Ranking locked: 28 unique compounds (64 pair rows)
  known_active compounds: 0


In [2]:
import pandas as pd
import json

# Deterministic sort: score desc, then cultivar_source asc within tie
final_table = final_table.sort_values(
    ["compound_id", "final_score", "cultivar_source"],
    ascending=[True, False, True],
).reset_index(drop=True)

# Duplication cap: top 2 pairs per compound (lexicographic cultivar tiebreak)
capped = final_table.groupby("compound_id", group_keys=False).head(2)

# Top 10 pairs by protocol §4.5 tiebreak:
# final_score desc → similarity desc → developability desc → inchikey asc → cultivar asc
shortlist = (
    capped.sort_values(
        ["final_score", "similarity_support", "developability_score",
         "inchikey", "cultivar_source"],
        ascending=[False, False, False, True, True],
    )
    .head(10)
    .copy()
)
shortlist["cultivar_selection_rule"] = "lexicographic_provenance"

# ---- Compound-level view: U = unique shortlist compounds (not screened universe)
shortlist_compound_view = (
    final_table[final_table["compound_id"].isin(shortlist["compound_id"].unique())]
    .sort_values("final_score", ascending=False)
    .drop_duplicates("compound_id")
    .copy()
)

cultivar_counts = (
    final_table.groupby("compound_id")
    .agg(
        all_cultivars=("cultivar_source", lambda x: "; ".join(sorted(set(x)))),
        n_cultivar_entries=("cultivar_source", lambda x: x.nunique()),
    )
    .reset_index()
)
shortlist_compound_view = shortlist_compound_view.drop(
    columns=["cultivar_source", "cultivar_evidence_level",
             "source_publication", "source_locator", "retrieval_rank"],
    errors="ignore",
)
shortlist_compound_view = shortlist_compound_view.merge(cultivar_counts, on="compound_id")
shortlist_compound_view["primary_cultivar"] = (
    shortlist_compound_view["all_cultivars"].apply(lambda s: s.split("; ")[0])
)

# ---- Save
shortlist.to_csv(ROOT / "results" / "primary_shortlist.csv", index=False)
shortlist_compound_view.to_csv(ROOT / "results" / "primary_shortlist_compound_level.csv", index=False)

# ---- Metadata: all counts at unique-compound level (protocol §2.1)
dedup_sl = shortlist.drop_duplicates("compound_id")

# ML tie-band diagnostics from the fresh ml_ad_scores
ml_ad_full = pd.read_csv(ROOT / "results" / "ml_ad_scores.csv")
band_counts = ml_ad_full["ML_rank_score"].value_counts()

metadata = {
    "N_star":                         int(len(shortlist)),
    "U_star":                         int(shortlist["compound_id"].nunique()),
    "shortlist_pair_n":               int(len(shortlist)),
    "screened_compound_n":            int(final_table["compound_id"].nunique()),
    "known_active_count":             int(dedup_sl["known_active"].sum()),
    "high_ml_ad_extrapolation_count": int(dedup_sl["high_ml_ad_extrapolation"].sum()),
    "ad_extrapolation_count":         int((dedup_sl["ad_flag"] == "extrapolation").sum()),
    "ad_missing_count":               0,
    "docking_missing_count":          0,
    "ml_rank_top_band_value":         float(band_counts.idxmax()),
    "ml_rank_top_band_count":         int(band_counts.max()),
    "ml_rank_top_band_share":         round(int(band_counts.max()) / len(ml_ad_full), 4),
    "ml_rank_unique_values_n":        int(ml_ad_full["ML_rank_score"].nunique()),
}
(ROOT / "results" / "shortlist_metadata.json").write_text(json.dumps(metadata, indent=2))

# --- Expose U_star/N_star for cell 4
N_star = metadata["N_star"]
U_star = metadata["U_star"]

print(f"Shortlist: N* = {N_star} pairs, U* = {U_star} unique compounds")
print(f"Screened universe: {metadata['screened_compound_n']} compounds")
print(f"ML tie-band: {metadata['ml_rank_top_band_count']}/{metadata['screened_compound_n']} "
      f"at {metadata['ml_rank_top_band_value']} ({metadata['ml_rank_top_band_share']*100:.1f}%)")
print("\nMetadata:")
print(json.dumps(metadata, indent=2))

Shortlist: N* = 10 pairs, U* = 5 unique compounds
Screened universe: 28 compounds
ML tie-band: 27/28 at 0.6333333333333333 (96.4%)

Metadata:
{
  "N_star": 10,
  "U_star": 5,
  "shortlist_pair_n": 10,
  "screened_compound_n": 28,
  "known_active_count": 0,
  "high_ml_ad_extrapolation_count": 0,
  "ad_extrapolation_count": 5,
  "ad_missing_count": 0,
  "docking_missing_count": 0,
  "ml_rank_top_band_value": 0.6333333333333333,
  "ml_rank_top_band_count": 27,
  "ml_rank_top_band_share": 0.9643,
  "ml_rank_unique_values_n": 2
}


In [3]:
import pandas as pd
import numpy as np

# 1. Get the Ligand-Only Shortlist
# We use the exact same capping and sorting logic, but on ligand_only_score
lig_table = final_table.sort_values("ligand_only_score", ascending=False)
lig_capped = lig_table.groupby("compound_id").head(2)
lig_shortlist = lig_capped.head(10)

# 2. Calculate Pair-Level Overlap (N*)
# A pair is (compound_id, cultivar_source)
locked_pairs = set(zip(shortlist["compound_id"], shortlist["cultivar_source"]))
lig_pairs = set(zip(lig_shortlist["compound_id"], lig_shortlist["cultivar_source"]))
pair_overlap = len(locked_pairs.intersection(lig_pairs))

# 3. Calculate Compound-Level Overlap (U*)
locked_compounds = set(shortlist["compound_id"])
lig_compounds = set(lig_shortlist["compound_id"])
compound_overlap = len(locked_compounds.intersection(lig_compounds))

# 4. Determine Sensitivity Label (Protocol §7.2)
N_star = len(shortlist)
if N_star == 0:
    label = "not_applicable_NO"
elif pair_overlap >= np.ceil(0.67 * N_star):
    label = "robust"
elif pair_overlap >= np.ceil(0.50 * N_star):
    label = "partially_robust"
else:
    label = "weight_sensitive"

print("--- Sensitivity Analysis ---")
print(f"Locked Shortlist N*: {N_star}")
print(f"Ligand-Only Shortlist N*: {len(lig_shortlist)}")
print(f"Pair-Level Overlap: {pair_overlap} / {N_star}")
print(f"Compound-Level Overlap: {compound_overlap} / {len(locked_compounds)}")
print(f"Sensitivity Label: {label.upper()}")

--- Sensitivity Analysis ---
Locked Shortlist N*: 10
Ligand-Only Shortlist N*: 10
Pair-Level Overlap: 10 / 10
Compound-Level Overlap: 5 / 5
Sensitivity Label: ROBUST


In [4]:
from pathlib import Path
import pandas as pd

cols = [
    "section", "row_id", "metric", "split", "value", "rank", "compound_id",
    "compound_name", "cultivar_source", "source_publication", "source_attempted",
    "structure_status", "reason", "final_score", "ligand_only_score",
    "ligand_only_rank", "failed_rules", "pattern", "pair_overlap",
    "sensitivity_label", "N_star", "U_star", "notes",
]

diag_rows = []

# A. Filtered but interesting — dedupe by compound before iterating
failed_dev = (
    final_table[final_table["developability_score"] < 1.0]
    .drop_duplicates("compound_id")
    .sort_values("final_score", ascending=False)
    .head(5)
)
for i, (_, row) in enumerate(failed_dev.iterrows(), 1):
    failed_rules = []
    if not row["lipinski"]:   failed_rules.append("Lipinski")
    if not row["veber"]:      failed_rules.append("Veber")
    if not row["pains"]:      failed_rules.append("PAINS")
    if not row["aggregator"]: failed_rules.append("Aggregator")
    diag_rows.append({
        "section":       "filtered_but_interesting",
        "row_id":        i,
        "compound_id":   row["compound_id"],
        "compound_name": row["compound_name"],
        "final_score":   row["final_score"],
        "failed_rules":  "; ".join(failed_rules),
        "reason":        "Failed developability",
    })

# B. Disagreement patterns — dedupe by compound
novel = (
    final_table[(final_table["ML_rank_score"] >= 0.80)
                & (final_table["similarity_support"] < 0.20)]
    .drop_duplicates("compound_id")
    .sort_values("final_score", ascending=False)
    .head(5)
)
for i, (_, row) in enumerate(novel.iterrows(), 1):
    diag_rows.append({
        "section":       "disagreement_patterns",
        "row_id":        i,
        "compound_id":   row["compound_id"],
        "compound_name": row["compound_name"],
        "pattern":       "high ML + low similarity",
        "notes":         "Novel scaffold, higher risk",
    })
if novel.empty:
    print("Note: disagreement_patterns is empty — no compounds meet ML_rank_score >= 0.80 "
          "under the corrected scores (all sit at 0.633333).")

# C. Sensitivity
diag_rows.append({
    "section":           "sensitivity",
    "row_id":            1,
    "pair_overlap":      pair_overlap,
    "sensitivity_label": label,
    "N_star":            N_star,
    "U_star":            U_star,
    "notes":             f"Compound overlap: {compound_overlap}/{U_star}",
})

# ---- Idempotent write: replace owned sections, preserve others (e.g. ledger)
diag_path = ROOT / "results" / "diagnostics.csv"
owned_sections = {"filtered_but_interesting", "disagreement_patterns", "sensitivity"}

if diag_path.exists():
    df_existing = pd.read_csv(diag_path)
    df_existing = df_existing[~df_existing["section"].isin(owned_sections)]
else:
    df_existing = pd.DataFrame(columns=cols)

df_new = pd.DataFrame(diag_rows, columns=cols)
df_final = pd.concat([df_existing, df_new], ignore_index=True).reindex(columns=cols)
df_final.to_csv(diag_path, index=False)

print(f"Diagnostics written: {len(df_new)} rows from this cell "
      f"({df_new['section'].value_counts().to_dict()})")

Note: disagreement_patterns is empty — no compounds meet ML_rank_score >= 0.80 under the corrected scores (all sit at 0.633333).
Diagnostics written: 5 rows from this cell ({'filtered_but_interesting': 4, 'sensitivity': 1})
